# backfill

> sessions that ran before the ledger existed

A hook only sees the next event. `panjika backfill` reads the transcripts of the harness, and the transcripts of its subagents. It writes the same records as a hook.

In [ ]:
#| default_exp backfill

In [ ]:
#| export
import json, os, re, shlex
from datetime import datetime
from pathlib import Path

from fastcore.foundation import L

from panjika.core import Home, hashed
from panjika.harness import act, plan
from panjika.core import action_for
from panjika.core import Scribe

## How to find and read a transcript

Claude Code makes one folder for each working directory under `~/.claude/projects`. The folder name is the path with the punctuation replaced. `read_recs` ignores the last line if it is incomplete, and a live transcript often ends with an incomplete line.

In [ ]:
#| export
SESSIONS = Path.home()/'.claude'/'projects'

def sess_dir(cwd=None, root=None):
    "The transcript folder for one project."
    if root: return Path(root)
    return SESSIONS/re.sub(r'[^a-zA-Z0-9]', '-', str(Path(cwd or '.').expanduser().resolve()))

def read_recs(path):
    "Every record in a transcript. It ignores an incomplete last line."
    out = L()
    try: raw = Path(path).read_text(encoding='utf-8', errors='replace')
    except OSError: return out
    for line in raw.splitlines():
        if not line.strip(): continue
        try: out.append(json.loads(line))
        except ValueError: continue
    return out

def at_of(rec):
    "The time of a record, in epoch seconds."
    t = (rec or {}).get('timestamp')
    if not t: return None
    try: return datetime.fromisoformat(str(t).replace('Z', '+00:00')).timestamp()
    except ValueError: return None

## How to read one record

`INJECTED` lists the openings of text that the harness added. A person did not type this text. Such text is often the first record in a transcript.

In [ ]:
#| export
INJECTED = ('<task-notification', '<system-reminder', '<wake ', '<local-command',
            '<command-name', '<user-prompt-submit-hook')

def is_prompt(rec):
    "Whether a record is a prompt. Its message content is text, not a tool result."
    return rec.get('type') == 'user' and isinstance((rec.get('message') or {}).get('content'), str)

def prompt_kind(rec, text):
    "Whether a person typed the text, or the harness added it."
    kind = (rec.get('origin') or {}).get('kind') if isinstance(rec.get('origin'), dict) else None
    if kind and kind != 'human': return 'injected'
    return 'injected' if str(text).lstrip().startswith(INJECTED) else 'human'

def blocks(msg):
    "A message's content blocks, always as a list."
    c = (msg or {}).get('content')
    return [] if c is None else ([c] if isinstance(c, str) else list(c))

def block_text(content):
    "The plain text of content blocks. A tool result is a string or a list of blocks."
    if isinstance(content, str): return content
    out = []
    for b in (content if isinstance(content, list) else [content]):
        if isinstance(b, str): out.append(b)
        elif isinstance(b, dict) and b.get('type') == 'text': out.append(b.get('text') or '')
        elif isinstance(b, dict) and b.get('type') != 'thinking' and 'text' in b: out.append(str(b['text']))
    return '\n'.join(x for x in out if x)

## Which files a call changed

`call_touches` reads the arguments and the result of one call.

`TOOL_PATHS` gives the argument that holds the path, for each tool. If a path has a character from `UNRESOLVED`, it is a pattern or a variable, and panjika ignores it.

A wrong path is worse than no path. An agent reads a wrong path as a file that an earlier session changed.

In [ ]:
#| export
TOOL_PATHS = {
    'Edit': ('file_path', 'edit'), 'Write': ('file_path', 'create'),
    'MultiEdit': ('file_path', 'edit'), 'NotebookEdit': ('notebook_path', 'edit'),
    'Read': ('file_path', 'read'), 'NotebookRead': ('notebook_path', 'read'),
}

UNRESOLVED = ('$', '*', '?', '{', '}', '[', ']')
_HEREDOC = re.compile(r'(?:^|\s)<<-?\s*[\x27\x22]?(\w+)[\x27\x22]?\s*$')
_SEPS = ('|', '||', '&&', ';', '&')
_REDIR = ('>', '>>', '2>', '2>>', '&>', '&>>')

def _cmd_lines(cmd):
    "The lines of a shell command, without the heredoc bodies."
    out, lines, i = [], cmd.splitlines(), 0
    while i < len(lines):
        out.append(lines[i])
        if (m := _HEREDOC.search(lines[i])):
            marker = m.group(1); i += 1
            while i < len(lines) and lines[i].strip() != marker: i += 1
        i += 1
    return out

def _segments(line):
    "One line, split at each shell operator."
    try: toks = shlex.split(line, comments=True)
    except ValueError: return []
    segs, cur = [], []
    for t in toks:
        if t in _SEPS: segs.append(cur); cur = []
        else: cur.append(t)
    return [s for s in segs + [cur] if s]

def _named(p):
    "One path, or `None` if panjika cannot name it."
    p = p.strip().strip('"').strip("'").rstrip(';&|')
    if not p or p in ('+', ';', '{}') or p.startswith(('-', '/dev/')): return None
    return None if any(c in p for c in UNRESOLVED) or p.isdigit() else p

def bash_paths(cmd):
    "The files that a shell command writes to. It ignores a path that it cannot name."
    if not cmd: return []
    out = []
    for line in _cmd_lines(cmd):
        for seg in _segments(line):
            if seg[0] in ('[[', '[', 'test'): continue
            for i, t in enumerate(seg):
                if t in _REDIR and i + 1 < len(seg): out.append(seg[i+1])
                elif len(t) > 1 and t[0] == '>' and t[1] != '&': out.append(t.lstrip('>'))
                elif len(t) > 2 and t[:2] in ('2>', '&>') and not t[2:].lstrip('>').startswith('&'):
                    out.append(t[2:].lstrip('>'))
            if seg[0] == 'tee':
                rest = [t for t in seg[1:] if not t.startswith('-')]
                if rest: out.append(rest[0])
            elif seg[0] == 'sed' and any(t == '-i' or t.startswith('-i.') for t in seg[1:]):
                out += [t for t in seg[1:] if not t.startswith('-')][1:]
    seen = []
    for p in map(_named, out):
        if p and p not in seen: seen.append(p)
    return seen

def call_touches(tool, args, result=None):
    "`[(path, action)]` for one call. Empty if the call changed no file that panjika can name."
    args, out = args or {}, []
    if tool in TOOL_PATHS:
        key, action = TOOL_PATHS[tool]
        if (p := args.get(key)): out.append((p, action))
    if isinstance(result, dict):
        for key in ('filePath', 'file_path', 'notebook_path'):
            if (p := result.get(key)) and not any(p == q for q, _ in out):
                out.append((p, 'read' if tool in ('Read', 'NotebookRead') else 'edit'))
    if tool == 'Bash': out += [(p, 'edit') for p in bash_paths(args.get('command'))]
    return out

def bodies(tool, args, result):
    "The file before and after a write, if the transcript has them."
    args, result = args or {}, result if isinstance(result, dict) else {}
    before, after = result.get('originalFile'), result.get('newFile')
    if after is None and tool == 'Write': after = args.get('content')
    return before, after

## One transcript as a plan

A backfill reads events that it recorded before. `rec_id` makes the id from the transcript, so a second backfill adds nothing.

A transcript can start in the middle of a conversation. Then panjika writes a note. It does not use a later prompt.

In [ ]:
#| export
def usage_of(recs):
    "The tokens in the assistant records. It counts each API response one time."
    seen, tot = set(), dict(input=0, output=0, cache=0)
    for r in recs:
        if r.get('type') != 'assistant': continue
        m = r.get('message') or {}
        mid = m.get('id') or r.get('requestId') or r.get('uuid')
        if mid in seen: continue
        seen.add(mid)
        u = m.get('usage') or {}
        tot['input']  += u.get('input_tokens') or 0
        tot['output'] += u.get('output_tokens') or 0
        tot['cache']  += (u.get('cache_read_input_tokens') or 0) + (u.get('cache_creation_input_tokens') or 0)
    return tot

def rec_id(*parts):
    "An id made from the transcript. A second read makes the same id."
    return 'b' + hashed('\x1f'.join(str(p) for p in parts), 22)


def transcript_plan(recs, session='', agent='', meta=None, harness='claude-code'):
    "One transcript as a plan of `Scribe` calls, in time order."
    recs = sorted(recs, key=lambda r: (at_of(r) or 0))
    first = next((r for r in recs if r.get('sessionId')), {})
    sid = session or first.get('sessionId') or ''
    cwd = next((r.get('cwd') for r in recs if r.get('cwd')), '')
    model = next(((r.get('message') or {}).get('model') for r in recs
                  if r.get('type') == 'assistant' and (r.get('message') or {}).get('model')), '')
    p = plan(f'{sid}:{agent}' if agent else sid, cwd)
    if not recs: return p
    stamps = [t for t in (at_of(r) for r in recs) if t]
    first_at, last_at = (stamps[0], stamps[-1]) if stamps else (None, None)
    act(p, 'begin', harness=harness, model=str(model or ''), agent=agent,
        parent=sid if agent else '', title=str((meta or {}).get('description') or ''),
        at=first_at, started=first_at, id=rec_id('begin', p.session))
    results = {b.get('tool_use_id'): (b, r) for r, b in _tool_blocks(recs)
               if b.get('type') == 'tool_result'}
    opened = False
    for r in recs:
        if is_prompt(r):
            text = r['message']['content']
            act(p, 'write', kind='session', prompt=text[:2000],
                origin=('agent' if agent else prompt_kind(r, text)), at=at_of(r),
                id=rec_id('prompt', p.session, r.get('uuid') or text[:80]))
            opened = True
            continue
        m = r.get('message')
        if not isinstance(m, dict): continue
        if not opened and (r.get('type') == 'assistant' or _has_tool(m)):
            act(p, 'note', text='resumed: the prompt for this work is not in this transcript',
                at=at_of(r), id=rec_id('resumed', p.session))
            opened = True
        _call_acts(p, r, m, results, p.session)
    u = usage_of(recs)
    act(p, 'end', status='imported', at=last_at, ended=last_at, id=rec_id('end', p.session),
        input=u['input'], output=u['output'], cache=u['cache'])
    return p

def _tool_blocks(recs):
    for r in recs:
        m = r.get('message')
        if isinstance(m, dict):
            for b in blocks(m):
                if isinstance(b, dict): yield r, b

def _has_tool(m):
    return any(isinstance(b, dict) and b.get('type') == 'tool_use' for b in blocks(m))

def _call_acts(p, rec, msg, results, sid=''):
    "The tool calls in one record, with their results and the files they changed."
    for b in blocks(msg):
        if not isinstance(b, dict) or b.get('type') != 'tool_use': continue
        res, rrec = results.get(b.get('id'), (None, None))
        payload = (rrec or {}).get('toolUseResult') if rrec else None
        payload = payload if isinstance(payload, dict) else None
        ok = not (res or {}).get('is_error')
        tool, args = str(b.get('name') or ''), (b.get('input') or {})
        tid = b.get('id') or f"{rec.get('uuid')}:{tool}"
        started, done = at_of(rec), at_of(rrec)
        act(p, 'step', tool=tool, target=_target_of(args), ok=ok, at=started,
            args=args, output=block_text((res or {}).get('content')) if res else '',
            id=rec_id('step', sid, tid), action=action_for(tool),
            secs=round(done - started, 2) if started and done else 0.0,
            summary='' if res else 'the call had not returned when this was read')
        if not (ok and res): continue
        before, after = bodies(tool, args, payload)
        for path, action in call_touches(tool, args, payload):
            if action == 'read': continue
            kw = dict(before=before, after_text=after) if after is not None else {}
            act(p, 'touch', path=path, action=action, at=at_of(rec),
                id=rec_id('touch', sid, tid, path), **kw)

def _target_of(args):
    for k in ('file_path', 'notebook_path', 'path', 'command', 'pattern', 'description'):
        if (v := (args or {}).get(k)): return str(v)[:160]
    return''

## How to apply a plan

If you give `home`, panjika uses it. If you do not, panjika uses the ledger of the folder where the session ran. `--all` needs this.

One bad act, or one bad transcript, does not stop the others.

In [ ]:
#| export
def apply_plan(p, home=None, start=None):
    "Run a plan against a ledger. Returns the number of records written."
    if not p.acts: return 0
    where = (p.start or start) if home is None else (start or p.start)
    sc = Scribe(home=home, session=p.session, start=where or '.')
    if not sc.home.exists: sc.home.init()
    for a in p.acts:
        a = dict(a); do = a.pop('do')
        if do == 'begin': getattr(sc, do)(**a); continue
        try: getattr(sc, do)(**a)
        except Exception: continue
    return len(p.acts)

def backfill_file(path, home=None, start=None, agent='', meta=None):
    "One transcript into the ledger. Returns `(session, records written)`."
    recs = read_recs(path)
    if not recs: return None, 0
    p = transcript_plan(recs, agent=agent, meta=meta)
    return p.session, apply_plan(p, home, start)

def backfill_session(path, home=None, start=None):
    "A transcript and every subagent transcript beside it."
    path = Path(path)
    out = [backfill_file(path, home, start)]
    subs = path.parent/path.stem/'subagents'
    for f in sorted(subs.glob('*.jsonl')) if subs.is_dir() else []:
        meta = None
        mp = f.with_suffix('.meta.json')
        if mp.exists():
            try: meta = json.loads(mp.read_text())
            except ValueError: meta = None
        out.append(backfill_file(f, home, start, agent=f.stem, meta=meta))
    return [(s, n) for s, n in out if s]

def backfill(cwd=None, root=None, home=None, start=None, limit=0):
    "Every Claude Code transcript for one project, or for every project."
    d = sess_dir(cwd, root)
    files = sorted(d.glob('*.jsonl')) if (cwd or root) else sorted(SESSIONS.glob('*/*.jsonl'))
    if limit: files = files[-int(limit):]
    out = []
    for f in files:
        try: out += backfill_session(f, home, start)
        except Exception: continue
    return out

## Tests

In [ ]:
from fastcore.test import test_eq

In [ ]:
test_eq(prompt_kind({}, 'hello'), 'human')
test_eq(prompt_kind({}, '<task-notification>x'), 'injected')
test_eq(prompt_kind({'origin': {'kind': 'task-notification'}}, 'hi'), 'injected')
test_eq(block_text([{'type': 'text', 'text': 'a'}, {'type': 'thinking', 'thinking': 'z'}]), 'a')

In [ ]:
#| echo: false
test_eq(call_touches('Edit',         {'file_path': 'a.py'}),        [('a.py', 'edit')])
test_eq(call_touches('Write',        {'file_path': 'a.py'}),        [('a.py', 'create')])
test_eq(call_touches('MultiEdit',    {'file_path': 'a.py'}),        [('a.py', 'edit')])
test_eq(call_touches('NotebookEdit', {'notebook_path': 'n.ipynb'}), [('n.ipynb', 'edit')])
test_eq(call_touches('Read',         {'file_path': 'a.py'}),        [('a.py', 'read')])
test_eq(call_touches('Bash', {'command': 'ls'}, {'filePath': 'made.py'}), [('made.py', 'edit')])
test_eq(call_touches('mcp__thing__do', {'target': 'a.py'}), [])

Claude Code writes one API response as several records, and each record repeats the same `usage`. If you add them, the total is about two times too large.

In [ ]:
for cmd, want, why in [
    ('echo hi > out.txt',                       ['out.txt'],     'a redirect'),
    ('echo x >>log.txt',                        ['log.txt'],     'an append with no space'),
    ('echo hi | tee -a out.log',                ['out.log'],     'tee writes its argument'),
    ("sed -i 's/a/b/' file.py",                 ['file.py'],     'sed in place'),
    ('sed -i.bak s/a/b/ f.py',                  ['f.py'],        'sed in place with a suffix'),
    ('sed -i s/a/b/ a.py b.py',                 ['a.py', 'b.py'], 'sed over several files'),
    ('cmd 2> err.log',                          ['err.log'],     'stderr redirected'),
    ('cmd &> all.log',                          ['all.log'],     'both streams redirected'),
    ('echo hi > "my file.txt"',                 ['my file.txt'], 'a quoted path with a space'),
    ('cat > f.py <<EOF\nprint(1 > 2)\nEOF',     ['f.py'],        'a heredoc body is data'),
    ('python -c "print(1 << 2)"\necho hi > o.txt', ['o.txt'],    'a shift is not a heredoc'),
    ('ls | grep x',                             [],              'a pipe writes no file'),
    ('cat a > /dev/null',                       [],              'the null device'),
    ('cmd 2>&1 | tail',                         [],              'a descriptor, not a file'),
    ('python3 -c "print(\'a > b\')"',            [],             'quoted, not a redirect'),
    ('[[ 5 > 3 ]] && echo yes',                 [],              'a comparison'),
    ('# echo hi > notes.txt',                   [],              'a comment'),
    ('for f in a b; do echo $f > /tmp/$f.txt; done', [],          'an unexpanded variable'),
    ('find . -name "*.py" -exec sed -i s/a/b/ {} +', [],          'no filename to name'),
]: assert bash_paths(cmd) == want, f'{why}: {cmd!r} gave {bash_paths(cmd)}'

recs = [{'type': 'assistant', 'message': {'id': 'm1', 'usage': {'output_tokens': 10}}},
        {'type': 'assistant', 'message': {'id': 'm1', 'usage': {'output_tokens': 10}}},
        {'type': 'assistant', 'message': {'id': 'm2', 'usage': {'output_tokens': 5}}}]
test_eq(usage_of(recs)['output'], 15)

## A session from before the ledger

An imported session gives the same accurate answer as a session that a hook recorded. An `Edit` gives the file before and after. A step gets its time from its own result.

In [ ]:
import subprocess, tempfile, uuid
from fastcore.test import test_eq
from panjika.core import Home
from panjika.git import landed
from panjika.core import Ledger

def _git(root, *a): subprocess.run(['git', *a], cwd=root, capture_output=True, check=True)

d = Path(tempfile.mkdtemp())/'proj'; d.mkdir(parents=True)
_git(d, 'init', '-q', '-b', 'main')
_git(d, 'config', 'user.email', 'a@b.c'); _git(d, 'config', 'user.name', 'Sam')
BEFORE = 'def total(x):\n    return sum(x)\n'
AFTER = 'def total(x):\n    x = [i for i in x if i >= 0]\n    return sum(x)\n'
(d/'charges.py').write_text(BEFORE)
_git(d, 'add', '-A'); _git(d, 'commit', '-qm', 'billing')

SID = 'aaaa1111-2222-3333-4444-555555555555'
def _rec(sec, **kw):
    return dict(uuid=uuid.uuid4().hex, sessionId=SID, cwd=str(d),
                timestamp=f'2026-01-15T09:00:{sec:02d}.000Z', **kw)

t = d.parent/f'{SID}.jsonl'
t.write_text('\n'.join(json.dumps(r) for r in [
    _rec(0, type='user', message={'role': 'user', 'content': 'make total() skip negatives'}),
    _rec(4, type='assistant', message={
        'id': 'm1', 'model': 'claude-opus-5', 'role': 'assistant', 'usage': {'output_tokens': 40},
        'content': [{'type': 'tool_use', 'id': 't1', 'name': 'Edit',
                     'input': {'file_path': 'charges.py'}}]}),
    _rec(9, type='user',
         toolUseResult={'filePath': 'charges.py', 'originalFile': BEFORE, 'newFile': AFTER},
         message={'role': 'user', 'content': [
             {'type': 'tool_result', 'tool_use_id': 't1', 'content': 'ok'}]}),
]) + '\n')

sid, n = backfill_file(t, home=Home(d/'.panjika'), start=d)
row = Ledger(d/'.panjika', d).session(sid)
test_eq((sid, row.harness, row.model), (SID, 'claude-code', 'claude-opus-5'))
test_eq(row.prompt, 'make total() skip negatives')
test_eq([f.path for f in row.files], ['charges.py'])
test_eq([s.secs for s in row.steps], [5.0])
assert 1768460000 < row.started < 1768560000

The commit holds the line of the session and the changes of a person. No commit holds only the lines of the session. The verdict is still accurate, because panjika compares the lines. A second backfill adds nothing.

In [ ]:
(d/'charges.py').write_text(AFTER + '\ndef fees(y):\n    return y * 2\n')
_git(d, 'add', '-A'); _git(d, 'commit', '-qm', 'skip negatives, and add fees()')
v = landed(SID, home=d/'.panjika', start=d)[0]
test_eq((v.state, v.kept, v.total, v.evidence), ('landed', 1, 1, 'lines'))

for _ in range(2): backfill_file(t, home=Home(d/'.panjika'), start=d)
row = Ledger(d/'.panjika', d).session(SID)
test_eq((row.n_steps, len(row.files)), (1, 1))